# 05. 보행 열스트레스 지수 (PHSI) — 과제 1 · '위해(장소 강도)' 층

규정: **LST 단독 금지** → 3축. **NDVI 강등**(그늘이 핵심, 녹색 아님):

`PHSI = 0.45·물리열(LST_peak) + 0.40·표면노출/무그늘(blockType) + 0.15·나무그늘(1−NDVI)`

- 표면노출: `src.heat_index.block_exposure` — 도로·광장·산업=무그늘 노출 최대, 녹지·수변=저.
- 각 축 robust min-max(2~98%). 가중치는 설계 선택(민감도 분석 대상).
- **이 지수는 '장소 강도'만.** 체류시간(비선형)·혼잡·이동은 노출층(과제2·06~09)에서 얹음. 나이 축 제외(혼잡 대리).
- ★진짜 그늘(건물그림자 SVF)은 건물 3D 필요 → 확보 시 축 추가.
- `peak_lowconf=1`(창릉·왕숙 갭필) 격자는 물리열 저신뢰 → 지수 잠정.

산출물: `data/results/heat_index.geojson`, `output/figures/fig07_phsi.png`

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, geopandas as gpd
from src.config import DATA_PROCESSED, DATA_RESULTS, OUT_FIGURES, CRS_SUBMIT, ZONE_COLORS
from src.heat_index import build_phsi

g = gpd.read_file(DATA_PROCESSED / "grid_features.geojson")
comp = build_phsi(g)          # heat, green_deficit, exposure, phsi
g = pd.concat([g, comp], axis=1)
g[["heat", "green_deficit", "exposure", "phsi"]].describe().round(3)

,heat,green_deficit,exposure,phsi
count,54029.000,54029.000,54029.000,54029.000
mean,0.461,0.499,0.525,0.492
std,0.267,0.275,0.335,0.221
min,0.000,0.000,0.050,0.026
25%,0.253,0.294,0.100,0.313
50%,0.427,0.518,0.600,0.510
75%,0.681,0.713,0.750,0.663
max,1.000,1.000,1.000,1.000


In [2]:
# 5등급 (분위수) — 취약 우선순위
labels = ["매우낮음", "낮음", "보통", "높음", "매우높음"]
g["phsi_level"] = pd.qcut(g["phsi"], 5, labels=labels).astype(str)   # Categorical -> str (gpd 0.13 호환)
print(g["phsi_level"].value_counts().sort_index())
print("\n지구별 평균 PHSI:")
print(g.groupby("zoneName")["phsi"].mean().round(3).sort_values(ascending=False))

phsi_level
낮음      10806
높음      10806
매우낮음    10806
매우높음    10806
보통      10805
Name: count, dtype: int64

지구별 평균 PHSI:
zoneName
성남분당             0.574
성남판교지구 택지개발사업    0.537
남양주왕숙 공공주택지구     0.445
고양창릉 공공주택지구      0.320
Name: phsi, dtype: float64


In [3]:
# 축별 기여 (개발 성남 vs 미개발 신도시) — 왜 뜨거운가의 분해
print(g.groupby("zoneName")[["heat", "green_deficit", "exposure", "phsi"]].mean().round(3))
# 최상위 취약 격자의 지배 축
top = g.nlargest(2000, "phsi")
print("\n최상위 2000 격자 평균 축:", top[["heat", "green_deficit", "exposure"]].mean().round(3).to_dict())

                heat  green_deficit  exposure   phsi
zoneName                                            
고양창릉 공공주택지구    0.120          0.487     0.481  0.320
남양주왕숙 공공주택지구   0.343          0.513     0.535  0.445
성남분당           0.610          0.517     0.556  0.574
성남판교지구 택지개발사업  0.606          0.457     0.490  0.537

최상위 2000 격자 평균 축: {'heat': 0.912, 'green_deficit': 0.748, 'exposure': 0.98}


In [4]:
DATA_RESULTS.mkdir(parents=True, exist_ok=True)
cols = ["grid_id", "zoneName", "blockType", "heat", "green_deficit", "exposure",
        "phsi", "phsi_level", "peak_lowconf", "geometry"]
g[cols].to_crs(CRS_SUBMIT).to_file(DATA_RESULTS / "heat_index.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
g.plot(column="phsi", cmap="YlOrRd", ax=axes[0], linewidth=0, legend=True, legend_kwds={"shrink": 0.6})
axes[0].set_title("보행 열스트레스 지수 (PHSI)")
g.plot(column="phsi_level", cmap="YlOrRd", ax=axes[1], linewidth=0, legend=True, categorical=True)
axes[1].set_title("PHSI 5등급")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig07_phsi.png", dpi=130); plt.close()